## Setup

### Import libs

In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path

import pandas as pd
from datasets import load_dataset, get_dataset_config_names, concatenate_datasets
import datasets

load_dotenv('.env')
HF_TOKEN = os.getenv("HF_TOKEN")

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = "1"

/home/amohammadi001/lid-ds/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### IO Util

In [2]:
download_dir = Path('data')
download_dir.mkdir(exist_ok=True)

schema_cols = [
    "text", "dataset", "code",
]
other_cols = ["config", "source_id", "script", "variant", "domain", "split"]

def prune_df(df):
  return df[[schema_cols]]

def prune_ds(ds):
  return ds.remove_columns(set(ds.column_names).intersection(other_cols))

def get_data_filename(name, sub_name=None, is_file=True):
    path = str(download_dir / f'{name}')
    if sub_name:
      path = os.path.join(path, sub_name)
    if is_file:
      path += '.parquet'
    return path

def save_data(data, name, code=None, df=False, ds=False):
  if isinstance(data, datasets.iterable_dataset.IterableDataset):
    pass
  elif ds or isinstance(data, datasets.Dataset):
      data = prune_ds(data)
  else:
    data = prune_df(data)

  path = get_data_filename(name, code)
  data.to_parquet(path)

def load_data(name, code=None, streaming=False):
  path = get_data_filename(name, code, is_file=False)
  if os.path.exists(path):
    return load_dataset(path, streaming=streaming)
  path = path + '.parquet'
  if os.path.exists(path):
    return pd.read_parquet(path)
  return None

def load_parquet_ds(filepath):
    return load_dataset(
        "parquet",
        data_files={"train": filepath},
        split="train",
    )

# used in glotcc v1
def normalize_glot_data(ds, ds_name, code, thr=0.80):
    ds = ds.filter(lambda x: x['identification-prob'] > thr and x['identification-consistency'] > thr)
    ds = ds.select_columns(['content', 'categories']).rename_columns({'content':'text', 'categories':'domain'})
    ds = ds.map(lambda x: {'dataset': ds_name, 'code': code})
    return ds

### Languages

In [17]:
SELECTED_LANGS = ['kur', 'fas', 'lki', 'tgk', 'oss', 'azb', 'tly', 'azj', 'prs', 'sdh', 'ckb', 'glk', 'urd', 'pus', 'azb', 'mzn', 'bal']

hard-coding the name and 'group' of the languages

In [18]:
iranian = {
    'bal_Arab': 'Balochi',
    'ckb_Arab': 'Central Kurdish',
    'prs_Arab': 'Dari',
    'diq_Latn': 'Dimli / Zazaki',
    'glk_Arab': 'Gilaki',
    'glk_Latn': 'Gilaki',
    'hac_Arab': 'Gurani / Gorani',
    'kiu_Latn': 'Kirmanjki',
    'kur_Arab': 'Kurdish',
    'lki_Arab': 'Laki',
    'mzn_Arab': 'Mazanderani',
    'kmr_Latn': 'Northern Kurdish',
    'lrc_Arab': 'Northern Luri',
    'oss_Cyrl': 'Ossetian',
    'pus_Arab': 'Pashto',
    'fas_Arab': 'Persian',
    'sdh_Arab': 'Southern Kurdish',
    'pbt_Arab': 'Southern Pashto',
    'tgk_Cyrl': 'Tajik',
    'tly_Latn': 'Talysh',
    'pes_Arab': 'Western Persian',
    }
indoaryan = {
    'kas_Arab': 'Kashmiri',
    'snd_Arab': 'Sindhi',
    'skr_Arab': 'Saraiki',
    'trw_Arab': 'Torwali',
    'urd_Arab': 'Urdu',
    }
semitic = {
    'acm_Arab': 'Mesopotamian Arabic',
    'arb_Latn': 'Standard Arabic',
    'arb_Arab': 'Standard Arabic',
    }
turkic = {
    'azj_Latn': 'North Azerbaijani',
    'azb_Arab': 'South Azerbaijani',
    'uzn_Latn': 'Northern_Uzbek',
    'uig_Arab': 'Uyghur',
    }
other = {
    'brh_Arab': 'Brahui'
}

lang_names = {**iranian, **indoaryan, **semitic, **turkic, **other}

lang_groups = {
    'Iranian': 'bal,ckb,diq,fas,glk,hac,kiu,kmr,kur,lki,lrc,mzn,oss,pbt,pes,prs,pus,sdh,tgk,tly',
    'Indo-Aryan': 'kas,skr,snd,trw,urd',
    'Semitic': 'acm,arb',
    'Turkic': 'azb,azj,uig,uzn',
    'Other': 'brh',
}
lang_groups = {k:v.split(',') for k,v in lang_groups.items()}

we should note that kur(kurdish),fas(persian),pus(pashto) are actually macro-languages.

In [19]:
get_group = lambda code: [k for k,v in lang_groups.items() if code in v][0]
get_script_name = lambda sc: {"Arab": "Arabic","Latn": "Latin","Cyrl": "Cyrillic"}.get(sc, sc)

rows = []
for config, name in lang_names.items():
  code, script = config.split('_')
  rows.append({
      'code': code, 'name': name,
      'script': get_script_name(script), 'group': get_group(code),
      'config': config,
  })

language_df = pd.DataFrame(rows).reset_index(drop=True)
language_df

,code,name,script,group,config
0,bal,Balochi,Arabic,Iranian,bal_Arab
1,ckb,Central Kurdish,Arabic,Iranian,ckb_Arab
2,prs,Dari,Arabic,Iranian,prs_Arab
3,diq,Dimli / Zazaki,Latin,Iranian,diq_Latn
4,glk,Gilaki,Arabic,Iranian,glk_Arab
5,glk,Gilaki,Latin,Iranian,glk_Latn
6,hac,Gurani / Gorani,Arabic,Iranian,hac_Arab
7,kiu,Kirmanjki,Latin,Iranian,kiu_Latn
8,kur,Kurdish,Arabic,Iranian,kur_Arab
9,lki,Laki,Arabic,Iranian,lki_Arab


### language config util

In [20]:
def get_lang_code(config: str):
  config = config.replace('-','_')
  c = config.replace('-','_').lower().split('_')
  if len(c) == 2:
    return c[0]
  return config

def get_lang_script(config: str):
  if get_lang_code(config) == config:
    return None
  return config.replace('-','_').split('_')[1]

def filter_langs(langs):
  return [x for x in langs if get_lang_code(x) in SELECTED_LANGS]

## Summary of huggingface datasets

In [30]:
hf_datasets = '''
cis-lmu/GlotCC-V1
cis-lmu/Glot500
cis-lmu/glotlid-corpus
openlanguagedata/flores_plus
openlanguagedata/oldi_seed
laurievb/open-lid-dataset
google/smol
'''.strip().split()

### util

In [ ]:
import os, re, json, requests
from pathlib import Path
from typing import Any

HF_DATASET_SERVER = "https://datasets-server.huggingface.co"


def hf_get(endpoint: str, params: dict[str, Any], token: str | None = None) -> dict[str, Any]:
    headers = {}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.get(f"{HF_DATASET_SERVER}/{endpoint}", params=params, headers=headers, timeout=60)
    r.raise_for_status()
    return r.json()


def guess_language_label(config = None, dataset= None, split = None):
  code = config.replace('-','_').split('_')[0]
  return code
def guess_language_script(config = None, dataset= None, split = None):
  script = config.replace('-','_').split('_')[1]
  return script


def summarize_hf_dataset(dataset: str, token: str | None = None) -> pd.DataFrame:
    response = hf_get("size", {"dataset": dataset}, token=token).get('size')

    data = []
    # data += [response.get('dataset')]
    if 'splits' in response:
      data += response.get('splits')
    else:
      data += response.get('configs')

    rows = []
    keys = [
        'config','num_rows','split',
        'num_bytes_parquet_files', 'num_bytes_memory',
        ]
    for item in data:
      rows.append({
          'dataset': dataset,
          **{k:item[k] for k in keys}
      })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    df = df.sort_values(
        by=["config", "split"],
    ).reset_index(drop=True)
    return df


def summarize_many_hf_datasets(datasets: list[str], token: str | None = None) -> pd.DataFrame:
    all_dfs = []
    for ds in datasets:
        try:
          df = summarize_hf_dataset(ds, token=token)
          if df.empty:
              all_dfs.append(pd.DataFrame([{
                  "dataset": ds,
                  "language_guess": None,
                  "config": None,
                  "split": None,
                  "num_rows": None,
                  "num_bytes": None,
                  "error": "No rows returned"
              }]))
          else:
              df["error"] = None
              all_dfs.append(df)
        except Exception as e:
            print(f'error with {ds}: {str(e)}')
            all_dfs.append(pd.DataFrame([{
                "dataset": ds,
                "language_guess": None,
                "config": None,
                "split": None,
                "num_rows": None,
                "num_bytes": None,
                "error": str(e)
            }]))
    return pd.concat(all_dfs, ignore_index=True)


def summarize_local_file(path: str) -> dict[str, Any]:
    """
    Very simple local summary.
    Supports csv, jsonl, parquet.
    """
    p = Path(path)
    suffix = p.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(p)
        return {"path": path, "num_rows": len(df), "num_cols": len(df.columns), "columns": list(df.columns)}

    if suffix in {".jsonl", ".ndjson"}:
        count = 0

def convert_bytes(b):
    if b is None:
        return None
    if b >= 2**30:
        return f"{b / 2**30:.2f} GB"
    return f"{b / 2**20:.2f} MB"


In [18]:
ds_df = summarize_many_hf_datasets(datasets, token=HF_TOKEN)
ds_df.iloc[0:2]

,dataset,config,num_rows,split,num_bytes_parquet_files,num_bytes_memory,error
0,cis-lmu/GlotCC-V1,aau-Latn,35,train,132506,219179,None
1,cis-lmu/GlotCC-V1,aaz-Latn,15,train,475176,867540,None


In [ ]:
summarize_hf_dataset('laurievb/open-lid-dataset', token=HF_TOKEN)

NameError: name 'summarize_hf_dataset' is not defined

### check dataset sizes

In [19]:
def clean_smol_rows(df):
  mask = df['dataset'] == 'google/smol'
  df.loc[mask, 'code'] = df.loc[mask, 'config'].map(lambda c: c.split('__')[1].split('_')[1])
  df = df[~df['config'].str.contains('gatitos', na=False)]
  return df

rep = ds_df.copy()
# postprocess some rows
rep['code'] = rep['config'].map(get_lang_code)

rep = clean_smol_rows(rep)
# only keep relevant languages
rep = rep[rep['code'].isin(SELECTED_LANGS+['default'])]
# more human reable sizes
rep[['size', 'memory']] = rep[['num_bytes_parquet_files','num_bytes_memory']].map(convert_bytes)

rep = rep.sort_values(by=['num_bytes_parquet_files'], ascending=[False])

rep['script'] = rep['config'].map(get_lang_script)
best_per_lang = rep.loc[rep.groupby('code')['num_bytes_parquet_files'].idxmax()]
best_per_lang = best_per_lang[['code', 'dataset', 'num_rows', 'size', 'script']]

top2_per_lang =  (
    rep.sort_values(['config', 'num_bytes_parquet_files'], ascending=[True, False])
       .groupby('config', group_keys=False)
       .head(2)
)

# columns to remove in the end
drop_cols = ['code', 'error', 'num_bytes_parquet_files', 'num_bytes_memory', 'script']
rep = rep.drop(columns=drop_cols)

In [20]:
best_per_lang

,code,dataset,num_rows,size,script
91,azb,cis-lmu/GlotCC-V1,1623,8.07 MB,Arab
94,azj,cis-lmu/GlotCC-V1,500000,2.83 GB,Latn
96,bal,cis-lmu/GlotCC-V1,352,1.06 MB,Arab
1316,ckb,cis-lmu/Glot500,855963,205.57 MB,Arab
255,default,cis-lmu/GlotCC-V1,684697321,2072.29 GB,NaN
311,fas,cis-lmu/GlotCC-V1,500000,2.17 GB,Arab
1363,glk,cis-lmu/Glot500,1354576,97.43 MB,Arab
1450,kur,cis-lmu/Glot500,405169,110.75 MB,Latn
606,lki,cis-lmu/GlotCC-V1,13,0.03 MB,Arab
1494,mzn,cis-lmu/Glot500,71710,7.68 MB,Arab


In [21]:
top2_per_lang

,dataset,config,num_rows,split,num_bytes_parquet_files,num_bytes_memory,error,code,size,memory,script
91,cis-lmu/GlotCC-V1,azb-Arab,1623,train,8459377,18104292,None,azb,8.07 MB,17.27 MB,Arab
1280,cis-lmu/Glot500,azb_Arab,1881,train,295931,584409,None,azb,0.28 MB,0.56 MB,Arab
1831,openlanguagedata/flores_plus,azb_Arab,1012,devtest,142757,397681,None,azb,0.14 MB,0.38 MB,Arab
93,cis-lmu/GlotCC-V1,azj-Cyrl,165,train,624286,1403248,None,azj,0.60 MB,1.34 MB,Cyrl
94,cis-lmu/GlotCC-V1,azj-Latn,500000,train,3035123759,5880963031,None,azj,2.83 GB,5.48 GB,Latn
1283,cis-lmu/Glot500,azj_Latn,1313831,train,121948833,210951307,None,azj,116.30 MB,201.18 MB,Latn
1833,openlanguagedata/flores_plus,azj_Latn,1012,devtest,136767,349176,None,azj,0.13 MB,0.33 MB,Latn
96,cis-lmu/GlotCC-V1,bal-Arab,352,train,1106368,2788782,None,bal,1.06 MB,2.66 MB,Arab
204,cis-lmu/GlotCC-V1,ckb-Arab,22023,train,80832747,177589425,None,ckb,77.09 MB,169.36 MB,Arab
1316,cis-lmu/Glot500,ckb_Arab,855963,train,215560457,438758381,None,ckb,205.57 MB,418.43 MB,Arab


In [22]:
rep

,dataset,config,num_rows,split,size,memory
255,cis-lmu/GlotCC-V1,default,684697321,train,2072.29 GB,3993.28 GB
2282,laurievb/open-lid-dataset,default,118296182,train,15.43 GB,19.01 GB
94,cis-lmu/GlotCC-V1,azj-Latn,500000,train,2.83 GB,5.48 GB
1605,cis-lmu/Glot500,tgk_Cyrl,11147510,train,2.36 GB,4.66 GB
311,cis-lmu/GlotCC-V1,fas-Arab,500000,train,2.17 GB,4.92 GB
1346,cis-lmu/Glot500,fas_Arab,18237931,train,1.62 GB,3.25 GB
1151,cis-lmu/GlotCC-V1,urd-Arab,221499,train,915.50 MB,1.95 GB
1634,cis-lmu/Glot500,urd_Arab,6008751,train,653.03 MB,1.33 GB
1316,cis-lmu/Glot500,ckb_Arab,855963,train,205.57 MB,418.43 MB
1067,cis-lmu/GlotCC-V1,tgk-Cyrl,50529,train,199.37 MB,443.63 MB


## Download datasets

### Flores_plus

#### flores language table

In [ ]:
# extracted from the flores article
cols = 'Code Language Script Family Subgrouping Res. Specification'.split()
langs = '''
ckb_Arab Central_Kurdish Arabic Indo-European Iranian Low _
kmr_Latn Northern_Kurdish Latin Indo-European Iranian Low _
pes_Arab Western_Persian Arabic Indo-European Iranian High _
prs_Arab Dari Arabic Indo-European Iranian Low Kabuli
pbt_Arab Southern_Pashto Arabic Indo-European Iranian Low Literary
urd_Arab Urdu Arabic Indo-European Indo-Aryan Low Lashkari
azb_Arab South_Azerbaijani Arabic Turkic Common_Turkic Low Tabrizi
acm_Arab Mesopotamian_Arabic Arabic Afro-Asiatic Semitic Low Baghdadi
arb_Arab Modern_Standard_Arabic Arabic Afro-Asiatic Semitic High _
arb_Latn Modern_Standard_Arabic Latin Afro-Asiatic Semitic Low _
azj_Latn North_Azerbaijani Latin Turkic Common_Turkic Low Shirvan
tgk_Cyrl Tajik Cyrlic Indo-Europian Iranian _ _
snd_Arab Sindhi Arabic Indo-European Indo-Aryan _ _
kas_Arab Kashmiri Arabic Indo-European Indo-Aryan _ _
'''
data = [line.split() for line in langs.strip().split('\n')]
flores200_df = pd.DataFrame(data, columns=cols)
flores200_df

,Code,Language,Script,Family,Subgrouping,Res.,Specification
0,ckb_Arab,Central_Kurdish,Arabic,Indo-European,Iranian,Low,_
1,kmr_Latn,Northern_Kurdish,Latin,Indo-European,Iranian,Low,_
2,pes_Arab,Western_Persian,Arabic,Indo-European,Iranian,High,_
3,prs_Arab,Dari,Arabic,Indo-European,Iranian,Low,Kabuli
4,pbt_Arab,Southern_Pashto,Arabic,Indo-European,Iranian,Low,Literary
5,urd_Arab,Urdu,Arabic,Indo-European,Indo-Aryan,Low,Lashkari
6,azb_Arab,South_Azerbaijani,Arabic,Turkic,Common_Turkic,Low,Tabrizi
7,acm_Arab,Mesopotamian_Arabic,Arabic,Afro-Asiatic,Semitic,Low,Baghdadi
8,arb_Arab,Modern_Standard_Arabic,Arabic,Afro-Asiatic,Semitic,High,_
9,arb_Latn,Modern_Standard_Arabic,Latin,Afro-Asiatic,Semitic,Low,_


#### download flores

In [ ]:
flores_repo = 'openlanguagedata/flores_plus'

def normalize_fb(dataset_name, ds):
  ds = ds.select_columns(["id", "iso_639_3", "iso_15924", "variant", "domain", "split", "text"])
  ds = ds.rename_column("iso_639_3", "code")
  ds = ds.rename_column("iso_15924", "script")
  ds = ds.ig("id", "source_id")
  ds = ds.map(lambda x: {'dataset': dataset_name})
  ds = ds.map(lambda x: {'script': x['script'].lower()})
  return ds

def get_flores(lang_codes, repo=flores_repo, split='dev'):
  return concatenate_datasets([
      normalize_fb('flores200', load_dataset(repo, code, split=split))
      for code in lang_codes
  ])

langs = get_dataset_config_names(flores_repo)
langs = filter_langs(langs)
flores = get_flores(langs)

flores_path = download_dir / 'flores.parquet'
flores.to_parquet(flores_path)
del flores

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 65.04ba/s]


In [36]:
pd.read_parquet(download_dir / 'flores.parquet').tail()

,source_id,code,script,variant,domain,split,text,dataset
5977,992,urd,arab,,wikivoyage,dev,ہل اسٹیشنوں پر سیاحت کا موسم عام طور پر ہندوست...,flores200
5978,993,urd,arab,,wikivoyage,dev,تاہم موسم سرما میں وہاں الگ ہی طرح کی خوبصورتی...,flores200
5979,994,urd,arab,,wikivoyage,dev,فقط چند ایک ایر لائنیں ہی سوگواری کے کرایوں کی...,flores200
5980,995,urd,arab,,wikivoyage,dev,جو ائرلائنس ان سہولیات کو پیش کرتی ہیں ان میں ...,flores200
5981,996,urd,arab,,wikivoyage,dev,تمام معاملوں میں، آپ کو لازماً ایئر لائن سے فو...,flores200


### Oldi

In [27]:
repo = 'openlanguagedata/oldi_seed'
dataset_name = repo.split('/')[1]

repo_langs = get_dataset_config_names(repo)
langs = filter_langs(repo_langs)

In [ ]:
ds_list = []

for lang_config in langs:
  ds = load_dataset(repo, lang_config)
  ds = ds['train']
  ds = ds.remove_columns([c for c in ds.column_names if c not in ["text","id"]])
  ds = ds.rename_column('id', 'source_id')
  code, script = lang_config.split('_')
  ds = ds.map(lambda x: {'dataset': dataset_name, 'code': code, 'script': script, 'config': lang_config, **x})
  ds_list.append(ds)

oldi = concatenate_datasets(ds_list)
save_data(oldi, dataset_name)
del oldi

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 13.60ba/s]


In [33]:
load_data(dataset_name)

,text,dataset,code
0,لیلیان دیانا گیش (14 اکتوبر 1893 – 27 فبروری 1...,oldi_seed,prs
1,گیش یک ستاره نامدار فلم از 1912 تا به دهه 1920...,oldi_seed,prs
2,او همچنان فعالیت قابل ملاحظه تلویزیونی از اوای...,oldi_seed,prs
3,چندین نسل نخست گیش‌ها وزراء دنکارد بودند.,oldi_seed,prs
4,مادر آنها ماجیستیک کیندی کیچین را باز کرد، و د...,oldi_seed,prs
...,...,...,...
6188,برای نشان دادن این، فرض کنید که ما عدد گویا n ...,oldi_seed,prs
6189,اگر 0 واقع نشود، بعد الگوریتم میتواند به بسیار...,oldi_seed,prs
6190,"""در ریاضیات، اعداد طبیعی آنهایی هستند که برای ...",oldi_seed,prs
6191,این زنجیره های پسوندها اعداد طبیعی را از نظر ک...,oldi_seed,prs


### glotlid-corpus

In [34]:
repo = 'cis-lmu/glotlid-corpus'
ds_name = 'glotlid'

configs = filter_langs(get_dataset_config_names(repo))
for config in configs:
  ds_dict = load_dataset(repo, config, token=HF_TOKEN)
  ds = ds_dict['train']

  # ds = ds.add_column("config", [config] * len(ds))
  code = get_lang_code(config)
  ds = ds.add_column("code", [code] * len(ds))
  ds = ds.add_column("dataset", [ds_name] * len(ds))

  save_data(ds, ds_name, code)

  del ds
  del ds_dict

README.md: 0.00B [00:00, ?B/s]

v3.1/azb_Arab/azb_Arab_GlotSparse.txt:   0%|          | 0.00/39.8M [00:00<?, ?B/s]

v3.1/azb_Arab/azb_Arab_Wikipedia.txt:   0%|          | 0.00/37.2M [00:00<?, ?B/s]

azb_Arab_cbc.txt: 0.00B [00:00, ?B/s]

azb_Arab_floresdev.txt: 0.00B [00:00, ?B/s]

...

urd_Latn_bhashaabhijnaanam.txt: 0.00B [00:00, ?B/s]

urd_Latn_cbc.txt: 0.00B [00:00, ?B/s]

urd_Latn_pbc.txt: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

In [35]:
load_data(ds_name, code)

,text,code,dataset
0,jazira rabaz,urd,glotlid
1,Jazeera Ghanam,urd,glotlid
2,jazeerah Nakhal,urd,glotlid
3,khazr,urd,glotlid
4,teesra: yani?,urd,glotlid
...,...,...,...
45144,us ne ek haftā aur intazār karke kabūtar ko du...,urd,glotlid
45145,jāsūsoṅ ke so jāne se pahle rāhab ne chhat par...,urd,glotlid
45146,yā kyā āp samajhte haiṅ ki kalām-e-muqaddas kī...,urd,glotlid
45147,"chunāṅche isrāīlī kan'āniyoṅ , hittiyoṅ , amor...",urd,glotlid


### Glot500

In [56]:
repo = 'cis-lmu/Glot500'
dsname = 'glot500'

configs = filter_langs(get_dataset_config_names(repo))

for config in configs:
  code = get_lang_code(config)

  ds = load_dataset(repo, config, split='train', streaming=True).take(20)
  ds = ds.map(
      lambda x: {
          'code': code,
        #   'config': config,
      })
  ds = ds.remove_columns(['lang_script'])
  save_data(ds, dsname, code)

  del ds

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

In [57]:
load_data(dsname, code)

,text,dataset,script,code
0,سعودی عرب کی عمرہ زائرین کے داخلے پر پابندی ،م...,MC4,Arab,tgk
1,معیار کے مواد گوکھا آسان پہلو زپ بندش Chic چھی...,MC4,Arab,tgk
2,نوٹس پی ڈبلیو اے کے لئے ایک مفت سافٹ ویئر ہے، ...,MC4,Arab,tgk
3,غزائم بلند ہیں – مگر و شائل کی کمی رکاوٹ بن کر...,MC4,Arab,tgk
4,لوڈشیڈنگ کے دوران پاکستانیوں کے 7 انوکھے کام,MC4,Arab,tgk
5,لاہور:نیند ایک ایسی چیز ہے جو انسانی صحت کے لی...,MC4,Arab,tgk
6,وزیراعظم پیروں کی اولاد ہیں لینے پر آئیں تو در...,MC4,Arab,tgk
7,پيام پيشين را پس از اينكه براي تخستين بار شما ...,MC4,Arab,tgk
8,تاہم فی الحال اسے صرف کروم 73 بیٹا اور کنیری م...,MC4,Arab,tgk
9,FMUSER FBE5 IPTV مرموزکار، مزید ایسے ایڈوب فلی...,MC4,Arab,tgk


### GlotCC V1

In [ ]:
from huggingface_hub import hf_hub_download

repo = 'cis-lmu/glotcc-v1'
ds_name = 'glotcc'
configs = filter_langs(get_dataset_config_names(repo))

for config in configs:
    config = config.replace('_','-')
    code = get_lang_code(config)

    file = hf_hub_download(
        repo_id="cis-lmu/GlotCC-V1",
        repo_type="dataset",
        filename=f"v1.0/{config}/{config}_0.parquet",
        local_dir="./downloads/glotcc ",
    )
    
    ds = load_parquet_ds(file)
    ds = normalize_glot_data(ds, ds_name, code, thr=0.9)
    save_data(ds, ds_name, code)
    del ds
    Path(file).unlink(missing_ok=True)

In [ ]:
ds = load_data('glotcc', 'fas')
del ds

### Push to the hub

In [ ]:
ds = load_dataset(
    "parquet",
    data_files={"train": "data/*/*.parquet"},
)